# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

import os
import chromadb
import json
from chromadb.utils import embedding_functions
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from lib.agents import Agent
from pydantic import BaseModel, Field
from lib.llm import LLM
from lib.state_machine import Run
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from lib.parsers import PydanticOutputParser
from chromadb.errors import NotFoundError
#from lib.vector_db import VectorStoreManager, CorpusLoaderService
#from lib.rag import RAG

In [ ]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
load_dotenv('/workspace/Code/.env')
assert os.getenv('OPENAI_API_KEY') is not None
assert os.getenv('TAVILY_API_KEY') is not None

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

CHROMA_PATH = ".chroma-db"
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(api_key=os.getenv("OPENAI_API_KEY"))
COLLECTION_NAME = "all-games-data"
chroma_client = chromadb.PersistentClient(CHROMA_PATH)
collection = chroma_client.get_collection(name=COLLECTION_NAME, embedding_function=embedding_fn)

from typing import List, Dict, Any

def make_retrieve_game_tool(collection):
    @tool
    def retrieve_game(query: str, n_results: int = 5) -> str:
        print("Tool: retrieve_game")
        print(f"DEBUG n_results = {n_results}")

        res = collection.query(
            query_texts=[query],
            n_results=n_results,
            include=["documents", "metadatas", "distances"]
        )

        ids = res.get("ids", [[]])[0] or []
        docs = res.get("documents", [[]])[0] or []
        metas = res.get("metadatas", [[]])[0] or []
        dists = res.get("distances", [[]])[0] or []

        out: List[Dict[str, Any]] = []
        for i in range(len(ids)):
            meta = metas[i] or {}
            out.append({
                "id": ids[i],
                "Name": meta.get("Name"),
                "Platform": meta.get("Platform"),
                "YearOfRelease": meta.get("YearOfRelease"),
                "Description": meta.get("Description"),
                "distance": dists[i] if i < len(dists) else None,
                "document": docs[i] if i < len(docs) else None,
            })

        return json.dumps(out)

    return retrieve_game


#### Evaluate Retrieval Tool

In [ ]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class EvaluationReport(BaseModel):
    useful: bool = Field(..., description="Whether the retrieved docs are sufficient to answer the question.")
    description: str = Field(..., description="Actionable explanation: what is missing / why sufficient / how to improve retrieval.")

@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> Dict[str, Any]:
    print("Tool: evaluate_retrieval")
    """
    Evaluates whether retrieved documents are sufficient to answer the user question.
    Explain what is missing, why sufficient or how to improve retrieval.

    args:
      - question: original question from user
      - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    returns:
      - dict with:
        - useful: bool
        - description: str (actionable)
    """
    docs_for_judge = retrieved_docs

    llm_judge = LLM(
        model="gpt-4o-mini", 
        temperature=0.0
    )

    judge_prompt = f"""
    You are an LLM judge. Decide if the retrieved documents are enough to respond to user question.
    Return your answer ONLY in the EvaluationReport format with fields:
    - useful: boolean
    - description: string

    User Question: {question}

    Retrieved Documents:
    {docs_for_judge}
    """
   
    # Default: allow answer if we retrieved anything at all
    response = llm_judge.invoke(
        input=judge_prompt,
        response_format=EvaluationReport
    )

    parser = PydanticOutputParser(model_class=EvaluationReport)
    evaluation = parser.parse(response)

    return {
        "useful": evaluation.useful,
        "description": evaluation.description
}


#### Game Web Search Tool

In [ ]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 


from tavily import TavilyClient
from typing import Any, Dict

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def game_web_search(question: str, max_results: int = 5) -> str:
    print("Tool: game_web_search")
    """
    Web search for game-industry questions when vector DB results are insufficient.
    args:
      - question: user question
      - max_results: number of results (default 5)
    returns:
      - JSON string: list of {title, url, content, score}
    """
    res = tavily.search(
        query=question,
        max_results=max_results,
        search_depth="advanced",
        include_answer=False
    )

    results = []
    for r in res.get("results", []):
        results.append({
            "title": r.get("title"),
            "url": r.get("url"),
            "content": r.get("content"),
            "score": r.get("score"),
        })

    return json.dumps(results)

### Agent

In [ ]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

retrieve_game = make_retrieve_game_tool(collection)

agent = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game,evaluate_retrieval,game_web_search],
    instructions=("""
    You are a helpful assistant for questions about video games.

    Follow this process exactly:
    1. Call retrieve_game with the user question.
    2. Store the result as `retrieved_docs`.
    3. Call evaluate_retrieval using:
    - question = the original user question
    - retrieved_docs = the output from retrieve_game
    4. If evaluation.useful is true:
    - Answer the question using ONLY retrieved_docs.
     5. If evaluation.useful is false:
    - show evaluation description 
    - Search information related to games only on the web by only using game_web_search
    - show information when returned
    - State the source of information. 
    
    Rules:
    - Never call evaluate_retrieval without retrieved_docs.
    - Never invent games.
    - If retrieve_game returns an empty list, ask one clarifying question.
    
    """.strip())
)

In [ ]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
# import time
# runner = Run(agent,start_timestamp=time.time())
# query = [
#     "When Pokémon Gold and Silver was released?",
#     "Which one was the first 3D platformer Mario game?",
#     "Was Mortal Kombat X realeased for Playstation 5?"

import time,re

queries = [
    "When was Pokémon Gold and Silver released?",
    "Which platform were they released on?",
    "Which country was the game first published in?"
]

session_id = "demo_session"

for i, q in enumerate(queries, 1):
    print(f"\n{'='*40}")
    print(f"Q{i}: {q}")
    print(f"{'='*40}")

    # Call agent (with or without session support)
    try:
        run = agent.invoke(q, session_id=session_id)
    except TypeError:
        run = agent.invoke(q)

    messages = run.get_final_state().get("messages", [])

    tools_used = []
    web_urls = []
    evaluation_result = None
    retrieval_preview = None

    for m in messages:
        # Capture retrieval output (preview only)
        if isinstance(m, ToolMessage) and getattr(m, "name", "") == "retrieve_game":
            try:
                docs = json.loads(m.content)
                retrieval_preview = docs[0] if docs else None
            except Exception:
                retrieval_preview = m.content
        # Capture evaluation result
        if isinstance(m, ToolMessage) and getattr(m, "name", "") == "evaluate_retrieval":
            try:
                evaluation_result = json.loads(m.content)
            except Exception:
                evaluation_result = m.content

        # What tools were used?
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                tools_used.append(getattr(tc, "name", None) or tc.function.name)

        # Capture web search URLs
        if isinstance(m, ToolMessage) and getattr(m, "name", "") == "game_web_search":
            try:
                results = json.loads(m.content)
                for r in results:
                    if r.get("url"):
                        web_urls.append(r["url"])
            except:
                pass

    # Final answer = last AI message without tool calls
    answer = next(
        (m.content for m in reversed(messages)
         if isinstance(m, AIMessage) and m.content and not getattr(m, "tool_calls", None)),
        "No answer"
    )

    # Also grab URLs written directly in the answer
    web_urls += re.findall(r"https?://\S+", answer)

    print("Answer:", answer)
    if evaluation_result:
        print("Evaluation (LLM Judge):")
        if isinstance(evaluation_result, dict):
            print(" - useful:", evaluation_result.get("useful"))
            print(" - description:", evaluation_result.get("description"))
        else:
            print(" - raw:", evaluation_result)
    print("Sources:")
    if "game_web_search" in tools_used:
        for url in web_urls[:3]:
            print(" -", url.rstrip(").,]"))
    else:
        print(" - Internal Game Database (ChromaDB)")


### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes